# Fault-diagnosis experiments — G01 to G03a

This notebook compares fault-type diagnosis studies. Diagnosis is evaluated after anomaly detection and is an aid to inspection, not an automatic maintenance decision.

## Study map

- **G01:** flat fault-type classification.
- **G02a–G02d:** hierarchical diagnosis and feature/weighting sensitivity.
- **G03a:** selected XGBoost instrumented-BMS diagnosis study.

G03a has access to an instrumented-BMS feature set and must not be described as symptom-only fault diagnosis.

In [8]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results' / 'final_runs'
STUDIES = [
    'G01_fault_type_classification',
    'G02a_hierarchical_baseline',
    'G02b_hierarchical_physics',
    'G02c_hierarchical_safe_context',
    'G02d_hierarchical_safe_context_balanced',
    'G03a_xgboost_instrumented_bms',
]

def latest_run(experiment):
    matches = sorted(RESULTS.glob(f'*_{experiment}'))
    if not matches:
        raise FileNotFoundError(f'No completed run found for {experiment}.')
    return matches[-1]

In [9]:
rows = []
for study in STUDIES:
    run_dir = latest_run(study)
    metrics = pd.read_csv(run_dir / 'metrics_summary.csv')
    metrics['study'] = study.split('_', 1)[0]
    rows.append(metrics)

comparison = pd.concat(rows, ignore_index=True)
core = comparison[comparison['metric'].str.contains('accuracy|f1', case=False, regex=True)]
display(core[['study', 'model', 'metric', 'folds', 'mean', 'median']].round(4))

,study,model,metric,folds,mean,median
0,G01,supervised_fault_type,accuracy,40,0.7180,0.7064
1,G01,supervised_fault_type,balanced_accuracy,40,0.7180,0.7064
2,G01,supervised_fault_type,macro_f1,40,0.2983,0.2924
5,G02a,supervised_hierarchical_fault_type,accuracy,40,0.7134,0.7085
6,G02a,supervised_hierarchical_fault_type,balanced_accuracy,40,0.7134,0.7085
7,G02a,supervised_hierarchical_fault_type,macro_f1,40,0.3184,0.3045
10,G02a,supervised_hierarchical_fault_type,coarse_accuracy,40,0.7713,0.7819
11,G02a,supervised_hierarchical_fault_type,coarse_macro_f1,40,0.4268,0.4284
12,G02a,supervised_hierarchical_fault_type,fault_type_accuracy,40,0.4268,0.4177
13,G02a,supervised_hierarchical_fault_type,fine_accuracy_given_correct_group,38,0.7851,0.9995


## Selected-study interpretation

The broad fault group is deliberately assessed separately from the exact fault type. Exact type should be interpreted only when the broad group is reliable; the operational reliability policy in O01 applies this rule.

In [10]:
g03a_run = latest_run('G03a_xgboost_instrumented_bms')
g03a_metrics = pd.read_csv(g03a_run / 'metrics_summary.csv')
selected_metrics = [
    'accuracy',
    'macro_f1',
    'coarse_accuracy',
    'coarse_macro_f1',
    'fault_type_accuracy',
]
display(g03a_metrics[g03a_metrics['metric'].isin(selected_metrics)].round(4))

for filename in ['coarse_confusion_matrix.csv', 'confusion_matrix.csv']:
    path = g03a_run / filename
    if path.exists():
        print(filename)
        display(pd.read_csv(path))

,model,metric,folds,mean,median
0,xgboost_hierarchical_fault_type,accuracy,40,0.7163,0.6858
2,xgboost_hierarchical_fault_type,macro_f1,40,0.3087,0.2927
5,xgboost_hierarchical_fault_type,coarse_accuracy,40,0.7761,0.7837
6,xgboost_hierarchical_fault_type,coarse_macro_f1,40,0.4426,0.4238
7,xgboost_hierarchical_fault_type,fault_type_accuracy,40,0.4326,0.3715


coarse_confusion_matrix.csv


,fold,true_fault_group,predicted_fault_group,rows
0,1,airside_damper,airside_damper,1378
1,1,airside_damper,heating_coil,62
2,1,airside_damper,normal,0
3,1,heating_coil,airside_damper,0
4,1,heating_coil,heating_coil,0
...,...,...,...,...
640,40,airside_damper,airside_damper,48
641,40,airside_damper,normal,0
642,40,normal,airflow_distribution,0
643,40,normal,airside_damper,0


confusion_matrix.csv


,fold,true_fault_type,predicted_fault_type,rows
0,1,exhaust_air_damper_stuck,exhaust_air_damper_stuck,517
1,1,exhaust_air_damper_stuck,heating_coil_valve_leak,62
2,1,exhaust_air_damper_stuck,normal,0
3,1,exhaust_air_damper_stuck,outdoor_air_damper_leak,485
4,1,exhaust_air_damper_stuck,outdoor_air_damper_stuck,376
...,...,...,...,...
1181,40,exhaust_air_damper_stuck,exhaust_air_damper_stuck,48
1182,40,exhaust_air_damper_stuck,normal,0
1183,40,normal,air_filter_blockage,0
1184,40,normal,exhaust_air_damper_stuck,0
